In [ ]:
# Refinamiento Wikidata

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (
    SparkSession.builder
    .appName("refinamiento_wikidata")
    .config("spark.driver.memory", "2g")
    .enableHiveSupport()
    .getOrCreate()
)

RAW_PATH = "/Obligatorio/landing/wikidata/festivals.csv"
REFINED  = "/Obligatorio/refined"

schema = T.StructType([
    T.StructField("festival",T.StringType()), T.StructField("festivalLabel",T.StringType()),
    T.StructField("countryLabel",T.StringType()), T.StructField("coord",T.StringType()),
    T.StructField("genreLabel",T.StringType()), T.StructField("setlist_fm_festival_ID",T.StringType()),
    T.StructField("identificador_MusicBrainz_de_serie",T.StringType()),
])
raw = spark.read.option("header",True).option("encoding","UTF-8").schema(schema).csv(RAW_PATH)

def norm(c): return F.lower(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
def country_key(c): return F.sha2(norm(c), 256)

# Limpieza: marcadores vacíos -> null, renombrado, extracción de id y coordenadas, validaciones
clean = raw
for col in clean.columns:
    clean = clean.withColumn(col, F.when(F.trim(F.col(col)) == "", None).otherwise(F.trim(F.col(col))))

clean = (
    clean
    .withColumnRenamed("festival","festival_uri")
    .withColumnRenamed("festivalLabel","festival_name")
    .withColumnRenamed("countryLabel","country")
    .withColumnRenamed("genreLabel","genre")
    .withColumnRenamed("setlist_fm_festival_ID","setlist_fm_id")
    .withColumnRenamed("identificador_MusicBrainz_de_serie","musicbrainz_series_id")
    .withColumn("wikidata_id", F.regexp_extract("festival_uri", r"/entity/(Q[0-9]+)", 1))
    .withColumn("longitude", F.regexp_extract("coord", r"Point\((-?[0-9.]+) (-?[0-9.]+)\)", 1).cast("double"))
    .withColumn("latitude",  F.regexp_extract("coord", r"Point\((-?[0-9.]+) (-?[0-9.]+)\)", 2).cast("double"))
    .withColumn("valid_id", F.col("wikidata_id").rlike("^Q[0-9]+$"))
    .withColumn("valid_coord", F.col("latitude").between(-90,90) & F.col("longitude").between(-180,180))
)

valid = clean.filter(
    F.col("festival_uri").isNotNull() & F.col("festival_name").isNotNull() &
    F.col("valid_id") & F.col("valid_coord")
)
print("Filas crudas:", raw.count(), "-> válidas:", valid.count())



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-24T00:00:19,936 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-06-24T00:00:21,097 WARN [Thread-4] org.apache.spark.util.Utils - Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Filas crudas: 1112 -> válidas: 1112


In [2]:
# Una coordenada representativa por festival (la más frecuente; en empate, orden alfabético)
coord_rank = valid.groupBy("wikidata_id","coord","latitude","longitude").agg(F.count("*").alias("n"))
w = Window.partitionBy("wikidata_id").orderBy(F.desc("n"), F.asc("coord"))
main_coord = coord_rank.withColumn("rn", F.row_number().over(w)).filter(F.col("rn")==1) \
                       .select("wikidata_id","latitude","longitude")

# Datos base por festival (un registro por festival)
festival_base = (
    valid.groupBy("wikidata_id","festival_uri","festival_name")
    .agg(
        F.first("country", ignorenulls=True).alias("country"),
        F.first("musicbrainz_series_id", ignorenulls=True).alias("musicbrainz_series_id"),
        F.first("setlist_fm_id", ignorenulls=True).alias("setlist_fm_id"),
    )
    .join(main_coord, "wikidata_id", "left")
)
print("Festivales distintos:", festival_base.count())

# Relación festival-género (normalizada, una fila por par)
festival_genres = (
    valid.filter(F.col("genre").isNotNull())
    .select("wikidata_id", F.lower(F.trim(F.col("genre"))).alias("genre_norm"),
            F.trim(F.col("genre")).alias("genre_label"))
    .dropDuplicates(["wikidata_id","genre_norm"])
)
print("Pares festival-género:", festival_genres.count())



Festivales distintos: 992
Pares festival-género: 312


In [3]:
# Nota: Wikidata no trae ciudad (city_id queda null), ni start_date ni capacity.
# country_id se deriva del nombre de país en español; se reconcilia con OpenFlights en la consolidación.
dim_festival = (
    festival_base.select(
        F.col("wikidata_id").alias("festival_id"),
        "wikidata_id",
        "festival_name",
        F.lit(None).cast("string").alias("city_id"),
        F.when(F.col("country").isNotNull(), country_key("country")).alias("country_id"),
        "latitude",
        "longitude",
        F.lit(None).cast("date").alias("start_date"),
        F.lit(None).cast("int").alias("capacity"),
        "musicbrainz_series_id",
        "setlist_fm_id",
    )
)
print("dim_festival:", dim_festival.count())
dim_festival.write.mode("overwrite").parquet(f"{REFINED}/dim_festival")



dim_festival: 992


In [4]:
# dim_genre (única fuente de géneros: Wikidata). genre_id determinístico.
dim_genre = (
    festival_genres
    .withColumn("genre_id", F.sha2(F.col("genre_norm"), 256))
    .groupBy("genre_id")
    .agg(F.first("genre_label").alias("genre_name"))
    .select("genre_id","genre_name")
)
print("dim_genre:", dim_genre.count())
dim_genre.write.mode("overwrite").parquet(f"{REFINED}/dim_genre")

# bridge_festival_genre
bridge_festival_genre = (
    festival_genres
    .withColumn("genre_id", F.sha2(F.col("genre_norm"), 256))
    .select(F.col("wikidata_id").alias("festival_id"), "genre_id")
    .dropDuplicates()
)
print("bridge_festival_genre:", bridge_festival_genre.count())
bridge_festival_genre.write.mode("overwrite").parquet(f"{REFINED}/bridge_festival_genre")



dim_genre: 102


bridge_festival_genre: 312


In [5]:
# Países de Wikidata (en español). Se unen a los de OpenFlights ya escritos.
# Se usa un path temporal para no leer y sobrescribir el mismo directorio en un solo job.
paises_wd = (
    valid.select("country").filter(F.col("country").isNotNull())
    .withColumn("country_id", country_key("country"))
    .groupBy("country_id").agg(F.first("country").alias("country_name"))
    .withColumn("country_iso", F.lit(None).cast("string"))
    .withColumn("source", F.lit("wikidata"))
    .select("country_id","country_name","country_iso","source")
)

prev = spark.read.parquet(f"{REFINED}/dim_country")
dim_country = prev.unionByName(paises_wd).dropDuplicates(["country_id"])

dim_country.write.mode("overwrite").parquet(f"{REFINED}/dim_country_tmp")
spark.read.parquet(f"{REFINED}/dim_country_tmp").write.mode("overwrite").parquet(f"{REFINED}/dim_country")
print("dim_country tras sumar Wikidata:", spark.read.parquet(f"{REFINED}/dim_country").count())



dim_country tras sumar Wikidata: 361


In [6]:
for t in ["dim_festival","dim_genre","bridge_festival_genre","dim_country"]:
    df = spark.read.parquet(f"{REFINED}/{t}")
    print(f"{t:24} filas={df.count():>6}  columnas={len(df.columns)}")

print("\nFestivales de EE. UU. (por país):")
(spark.read.parquet(f"{REFINED}/dim_festival")
    .filter(F.lower(F.col("country_id")).isNotNull())  # placeholder; el filtro real es por país
    .select("festival_name","latitude","longitude").show(5, truncate=False))

print("Géneros más frecuentes:")
(spark.read.parquet(f"{REFINED}/bridge_festival_genre")
    .groupBy("genre_id").count().orderBy(F.desc("count"))
    .join(spark.read.parquet(f"{REFINED}/dim_genre"), "genre_id")
    .select("genre_name","count").show(10, truncate=False))



dim_festival             filas=   992  columnas=11
dim_genre                filas=   102  columnas=2
bridge_festival_genre    filas=   312  columnas=2
dim_country              filas=   361  columnas=4

Festivales de EE. UU. (por país):
+----------------------+-----------+-----------+
|festival_name         |latitude   |longitude  |
+----------------------+-----------+-----------+
|Fesztergom            |47.79327778|18.73430556|
|Q1001859              |62.1948    |-6.8558    |
|Burg-Herzberg-Festival|50.7667    |9.51667    |
|Exit                  |45.25      |19.86666679|
|Proms                 |51.501111  |-0.1775    |
+----------------------+-----------+-----------+
only showing top 5 rows

Géneros más frecuentes:
+------------------+-----+
|genre_name        |count|
+------------------+-----+
|rave              |2    |
|show de televisión|1    |
|canción de bardo  |1    |
|metal extremo     |1    |
|death metal       |1    |
|música colombiana |1    |
|thrash metal      |1    |
|Fut